# Radon in domestic wells: building the analysis dataset

This notebook combines three files from the USGS Circular 1360 data archive (DeSimone, McMahon and Rosen, 2014, *Water quality in principal aquifers of the United States, 1991–2010*) into one pandas DataFrame with one row per well:

- `wellsites`: aquifer, lithology, NAWQA study code, water use, location
- `wellchars`: well depth, depth to water, confinement
- `radon`: radon-222 concentration (pCi/L), remark code, 2-sigma uncertainty

The final subset is domestic wells sampled in the NAWQA major-aquifer studies (study type `sus`), with censoring information prepared for a censored likelihood (e.g. `cens()` in brms).

In [ ]:
from pathlib import Path
import io
import numpy as np
import pandas as pd

# Folder containing the archive files. Change if needed.
DATA_DIR = Path("well_data")

# File names as uploaded (the original archive uses dots, e.g. pasum_pub.radon.txt)
FILES = {
    "sites": "pasum_pub_wellsites.txt",
    "chars": "pasum_pub_wellchars.txt",
    "radon": "pasum_pub_radon.txt",
}

## Reading the archive files

Each file is tab-separated and has the same layout:

1. a metadata header where every line starts with `#`,
2. a line of column names,
3. a line of format codes such as `10s` or `12n` (string / numeric width), which is not data.

Measurement columns come in pairs: `R_<code>_<name>` holds the remark code (`-` = detected value, `<` = less than the reporting level, `E` = estimated) and `V_<code>_<name>` holds the value. Everything is read as text first so that identifiers and codes are not altered, and numeric columns are converted explicitly afterwards.

In [ ]:
def read_archive(path):
    """Read a NAWQA archive text file into a DataFrame of strings."""
    text = Path(path).read_text(encoding="latin-1").replace("\r", "")
    lines = [ln for ln in text.split("\n")
             if ln.strip() and not ln.startswith("#") and not ln.strip().startswith("---")]
    header, body = lines[0], lines[2:]   # lines[1] is the format-code line
    df = pd.read_csv(io.StringIO("\n".join([header] + body)), sep="\t", dtype=str)
    return df.apply(lambda s: s.str.strip())

sites = read_archive(DATA_DIR / FILES["sites"])
chars = read_archive(DATA_DIR / FILES["chars"])
radon = read_archive(DATA_DIR / FILES["radon"])

for name, df in [("sites", sites), ("chars", chars), ("radon", radon)]:
    print(f"{name:6s} rows={len(df):5d}  unique wells={df.well_identifier.nunique():5d}")

All three files have one row per well, so `well_identifier` can be used as a one-to-one join key. The check below stops the notebook if that ever stops being true (for example with a different version of the archive).

In [ ]:
for name, df in [("sites", sites), ("chars", chars), ("radon", radon)]:
    assert df.well_identifier.is_unique, f"duplicate wells in {name}"
radon.columns.tolist()

## Selecting and renaming columns

Only the columns needed for modeling are kept, with shorter names. The radon minimum detectable concentration (parameter 99327) is reported for only a few hundred samples, so it is kept for reference but not used directly.

In [ ]:
sites_sel = sites[[
    "well_identifier", "principal_aquifer_lithology", "principal_aquifer", "glacial_region",
    "primary_water_use", "nawqa_sucode1", "nawqa_suc1_type",
    "state_postal_code", "county",
]].rename(columns={
    "principal_aquifer_lithology": "lithology",
    "nawqa_sucode1": "study",
    "nawqa_suc1_type": "study_type",
    "primary_water_use": "water_use",
    "state_postal_code": "state",
})

chars_sel = chars[["well_identifier", "well_depth", "V_72019_depth_to_water", "confinement"]].rename(
    columns={"well_depth": "well_depth_ft", "V_72019_depth_to_water": "depth_to_water_ft"})

radon_sel = radon[[
    "well_identifier", "sampling_date",
    "R_82303_Radon-222", "V_82303_Radon-222",
    "V_76002_Radon-222_2-sigma_combined_uncertainty",
    "V_99327_Radon-222_sample-specific_minimum_detectable_concentration",
]].rename(columns={
    "R_82303_Radon-222": "radon_remark",
    "V_82303_Radon-222": "radon",
    "V_76002_Radon-222_2-sigma_combined_uncertainty": "radon_unc_2sigma",
    "V_99327_Radon-222_sample-specific_minimum_detectable_concentration": "radon_mdc",
})

## Joining

Sites is the base table. Well characteristics are added with a left join (a well without them keeps missing values). Radon is joined with an inner join, since wells without a radon measurement are of no use for this analysis.

In [ ]:
wells = (sites_sel
         .merge(chars_sel, on="well_identifier", how="left", validate="one_to_one")
         .merge(radon_sel, on="well_identifier", how="inner", validate="one_to_one"))

num_cols = ["well_depth_ft", "depth_to_water_ft", "radon", "radon_unc_2sigma", "radon_mdc"]
wells[num_cols] = wells[num_cols].apply(pd.to_numeric, errors="coerce")
wells["sampling_date"] = pd.to_datetime(wells["sampling_date"].str[:10], errors="coerce")
wells["year"] = wells["sampling_date"].dt.year

wells = wells[wells.radon.notna()].copy()
print(len(wells), "wells with a radon value")
wells.head()

## Subsetting to domestic wells in major-aquifer studies

The archive also contains public-supply wells and monitoring wells from land-use studies (study types `lusag` and `lusur`), which sample shallow groundwater under farmland or urban areas. Those are different populations, so the analysis uses domestic wells from the major-aquifer studies only. Wells with an unknown aquifer are dropped because they cannot be placed in the hierarchy.

In [ ]:
print(pd.crosstab(wells.water_use, wells.study_type, margins=True))

dom = wells[(wells.water_use == "DOMESTIC")
            & (wells.study_type == "sus")
            & (wells.principal_aquifer != "Unknown or other")].copy()
print("\nselected wells:", len(dom))

## Grouping variables

The archive stores the glacial aquifer system as a single principal aquifer. The USGS reports split it into regions, and the `glacial_region` column allows the same here. Integer indices (starting at 1, as Stan expects) are added for each grouping level.

In [ ]:
dom["aquifer"] = np.where(
    dom.principal_aquifer == "Glacial",
    "Glacial - " + dom.glacial_region.fillna("unknown"),
    dom.principal_aquifer,
)

for col in ["lithology", "aquifer", "study"]:
    dom[col + "_idx"] = dom[col].astype("category").cat.codes + 1

print("lithologies:", dom.lithology.nunique(),
      " aquifers:", dom.aquifer.nunique(),
      " studies:", dom.study.nunique())

### Is the hierarchy strictly nested?

A nested model (studies within aquifers within lithologies) assumes each study belongs to exactly one aquifer and each aquifer to exactly one lithology. If not, crossed group effects are needed instead.

In [ ]:
aq_per_study = dom.groupby("study").aquifer.nunique()
lith_per_aq = dom.groupby("aquifer").lithology.nunique()
print("studies spanning >1 aquifer:", (aq_per_study > 1).sum())
print("aquifers spanning >1 lithology:", (lith_per_aq > 1).sum())
aq_per_study[aq_per_study > 1]

## Censoring

Two kinds of values cannot be treated as ordinary observations:

- **Remark `<`**: the value is below the reporting level. In this archive the value column then holds the reporting level itself, which is the censoring limit $L_i$.
- **Zero or negative values**: radon counts are background-corrected, so a sample with almost no radon can come out at or below zero. These are treated as left-censored at a low limit. The median sample-specific minimum detectable concentration in the archive is used for this (about 25 pCi/L).

For a model on the log scale, the response and the limit are both logged. For censored rows, `log_radon` is set to the log of the limit, which is the convention brms uses with `cens(censored)` where `censored` is `"left"` or `"none"`.

In [ ]:
mdc_default = float(radon_sel["radon_mdc"].pipe(pd.to_numeric, errors="coerce").median())
print("median minimum detectable concentration:", mdc_default)

below_rl = dom.radon_remark.eq("<")
nonpos = dom.radon.le(0) & ~below_rl

dom["censored"] = np.where(below_rl | nonpos, "left", "none")
dom["cens_limit"] = np.select([below_rl, nonpos], [dom.radon, mdc_default], default=np.nan)
dom["radon_for_model"] = np.where(dom.censored == "left", dom.cens_limit, dom.radon)
dom["log_radon"] = np.log(dom.radon_for_model)

print(dom.censored.value_counts())
dom.loc[dom.censored == "left",
        ["well_identifier", "aquifer", "radon_remark", "radon", "radon_unc_2sigma", "cens_limit"]]

## Summary of the final dataset

In [ ]:
print(dom.radon.describe(percentiles=[0.1, 0.5, 0.9]).round(1))

summary = (dom.groupby(["lithology", "aquifer"])
             .agg(n_wells=("radon", "size"), median_radon=("radon", "median"))
             .round(0))
summary

In [ ]:
print("missing well depth:", dom.well_depth_ft.isna().sum())
print("missing confinement:", dom.confinement.isna().sum())
print("aquifers with fewer than 5 wells:", (dom.groupby("aquifer").size() < 5).sum())

## Save

In [ ]:
out_cols = [
    "well_identifier", "state", "county", "sampling_date", "year",
    "lithology", "aquifer", "study", "lithology_idx", "aquifer_idx", "study_idx",
    "well_depth_ft", "depth_to_water_ft", "confinement",
    "radon_remark", "radon", "radon_unc_2sigma",
    "censored", "cens_limit", "radon_for_model", "log_radon",
]
dom[out_cols].to_csv("domestic_well_radon.csv", index=False)
print("saved", len(dom), "rows")